# Pareidolia ML - Memoria del Proyecto

Este notebook documenta el resumen del proyecto completo de Machine Learning para la detección de Pareidolia (ilusión óptica de caras en objetos).

## 1. Estructura del Proyecto

```
Pareidolia_ML/
├── src/
│   ├── utils/                    # Módulos reutilizables
│   │   ├── constants.py          # Configuración y constantes
│   │   ├── data_loader.py        # Carga y procesamiento de datos
│   │   ├── model_builder.py      # Construcción de modelos
│   │   ├── training.py           # Funciones de entrenamiento
│   │   ├── evaluation.py         # Evaluación y métricas
│   │   └── prediction.py         # Predicciones
│   ├── model/                    # Modelos entrenados
│   │   ├── production/           # Modelo de producción final
│   │   ├── Xception.keras        # Modelo base
│   │   ├── Xception_finetuned.keras
│   │   ├── Xception_augmented.keras
│   │   └── Xception_augmented_finetuned.keras
│   └── notebooks/                # Notebooks ejecutables
│       ├── 01_data_preparation.ipynb
│       ├── 02_model_comparison.ipynb
│       ├── 03_enhance_models.ipynb
│       └── 04_predictions.ipynb
├── data/                         # Datos del proyecto
│   ├── data.npz                  # Datos normalizados
│   ├── data_gray.npz             # Datos en escala de grises
│   ├── data_aug.npz              # Datos aumentados
│   ├── train/                    # Datos originales de entrenamiento
│   │   ├── cara/
│   │   └── sin-cara/
│   ├── test/                     # Datos de test
│   │   ├── cara/
│   │   └── sin-cara/
│   └── predictions/              # Predicciones y visualizaciones
│       └── grad_cam/
├── resources/
│   └── img/                      # Imágenes y recursos visuales
└── README.md
```

## 2. Descripción del Proyecto

**Objetivo:** Construir un clasificador de imágenes capaz de detectar si una imagen contiene una pareidolia (ilusión óptica de una cara).

**Clases:**
- `cara`: Imágenes con pareidolia (cara detectada)
- `sin-cara`: Imágenes sin pareidolia (sin cara detectada)

**Enfoque:** Transfer Learning con arquitecturas preentrenadas (EfficientNetB0, ResNet50, Xception)

## 3. Paso 1: Preparación de Datos

**Archivo:** `src/notebooks/01_data_preparation.ipynb`

### Actividades:
1. **Carga de imágenes:** Lectura desde `data/train/` y `data/test/`
2. **Redimensionamiento:** A 224×224 píxeles
3. **Normalización:** Valores entre 0 y 1 (división entre 255)
4. **Barajado:** Mezcla aleatoria de datos de entrenamiento
5. **Guardado:** Serialización en `.npz` para carga rápida

### Resultados:
- Archivo: `data/data.npz`
- Distribución de clases verificada y equilibrada

## 4. Paso 2: Comparación de Modelos

**Archivo:** `src/notebooks/02_model_comparison.ipynb`

**Modelo guardado:**
- `src/model/Xception.keras` - Modelo base entrenado

### Modelos Probados:
1. **EfficientNetB0** - Arquitectura eficiente, entrada 224×224
2. **ResNet50** - Arquitectura robusta, entrada 224×224
3. **Xception** - Especializado en texturas, entrada 299×299

### Arquitectura Común:
```
Backbone (congelado) → GlobalAveragePooling2D → 
BatchNormalization → Dense(128, relu) → Dropout(0.4) → 
Dense(1, sigmoid)
```

### Parámetros de Entrenamiento:
- Optimizer: Adam (lr=1e-4)
- Loss: Binary Crossentropy
- Metrics: Accuracy
- Callbacks: Early Stopping, ReduceLROnPlateau

### Resultado:
**✓ Mejor modelo:** Xception con AUC-ROC más alto

## 5. Paso 3: Mejora de Modelos

**Archivo:** `src/notebooks/03_enhance_models.ipynb`

### Proceso de Mejora:
1. **Fine-tuning:** Cargar modelo Xception entrenado, descongelar últimas capas, recompilar con learning rate bajo (1e-5)
2. **Data Augmentation:** Aumentar datos de entrenamiento con rotaciones, zoom, desplazamientos
3. **Combinación:** Entrenar con ambas técnicas para mejores resultados

### Mejoras Aplicadas:
- Fine-tuning: Adaptación más fina a características específicas del dataset
- Augmentation: Mayor robustez ante variaciones de imágenes
- Combinado: Mejor rendimiento general

### Modelos Guardados:
- `src/model/Xception_finetuned.keras` - Con fine-tuning
- `src/model/Xception_augmented.keras` - Con data augmentation
- `src/model/Xception_augmented_finetuned.keras` - Combinado (mejor rendimiento)

## 6. Paso 4: Predicciones y Validación

**Archivo:** `src/notebooks/04_predictions.ipynb`

### Funcionalidades:
- Predicción en imágenes individuales
- Predicciones en lote
- Visualización de resultados
- Evaluación en dataset de test

### Métricas Reportadas:
- Accuracy total
- Accuracy por clase
- Matriz de confusión
- Curva ROC

## 7. Módulos Principales (`src/utils/`)

### Importación:
```python
from src.utils import (
    read_data,
    load_and_prepare_data,
    load_data_npz,
    build_xception,
    train_model,
    evaluate_model,
    predict_single_image,
    batch_predict
)
```

### 7.1 constants.py
- Configuración centralizada (dimensiones, rutas, hiperparámetros)
- `IMAGE_SIZE = (224, 224, 3)`
- Rutas a: `data/`, `src/model/`, datasets `.npz`

### 7.2 data_loader.py
- `read_data()`: Carga imágenes desde carpeta
- `load_and_prepare_data()`: Pipeline completo de carga
- `save_data_npz()`: Serialización a .npz
- `load_data_npz()`: Deserialización desde .npz
- `preprocess_image()`: Procesamiento individual de imágenes

### 7.3 model_builder.py
- `build_xception()`: Constructor especializado para Xception
- `build_efficient_net_b0()`, `build_resnet50()`
- `freeze_backbone()`, `unfreeze_backbone_layers()`
- `augmentar_dataset()`: Data augmentation en tiempo real

### 7.4 training.py
- `train_model()`: Entrenamiento básico con callbacks
- `train_model_custom()`: Con validation set personalizado
- `train_with_augmentation()`: Con data augmentation

### 7.5 evaluation.py
- `evaluate_model()`: Evaluación completa en test set
- `plot_confusion_matrix()`, `plot_learning_curves()`, `plot_roc_curve()`
- `find_optimal_threshold()`: Búsqueda de threshold óptimo

### 7.6 prediction.py
- `predict_single_image()`: Predicción en una imagen
- `batch_predict()`: Predicciones en lote
- `visualize_predictions()`: Grilla de predicciones visuales

## 8. Modelos Disponibles

**Ubicación:** `src/model/` (múltiples versiones disponibles)

### Especificaciones Comunes:
- **Arquitectura:** Xception con Transfer Learning
- **Entrada:** 299×299×3
- **Salida:** Probabilidad binaria (0-1)
- **Threshold:** 0.5

### Modelos Entrenados:
1. **`Xception.keras`** - Modelo base sin fine-tuning
2. **`Xception_finetuned.keras`** - Con fine-tuning de capas superiores
3. **`Xception_augmented.keras`** - Entrenado con data augmentation
4. **`Xception_augmented_finetuned.keras`** - Mejor rendimiento (recomendado para producción)

### Uso:
```python
import tensorflow as tf
from src.utils import predict_single_image, batch_predict

# Cargar modelo recomendado
model = tf.keras.models.load_model('src/model/Xception_augmented_finetuned.keras')

# Predicción en una imagen
result = predict_single_image(model, 'ruta/imagen.jpg')

# Predicciones en lote
batch_results = batch_predict(model, image_paths)
```

## 9. Cómo Usar los Notebooks

### Secuencia Recomendada:
1. **01_data_preparation.ipynb** → Prepara los datos y genera `data/data.npz`
2. **02_model_comparison.ipynb** → Compara modelos y genera `Xception.keras`
3. **03_enhance_models.ipynb** → Mejora el mejor modelo (fine-tuning + augmentation)
4. **04_predictions.ipynb** → Valida en dataset de test

### Requisitos:
- Python 3.8+
- TensorFlow >= 2.10
- Numpy, Pandas, Matplotlib, Scikit-learn

### Instalación:
```bash
pip install -r requirements.txt
```

### Cargar datos rápidamente:
```python
from src.utils import load_data_npz

X_train, y_train, X_test, y_test = load_data_npz('data/data.npz')
```

## 10. Conclusiones y Próximos Pasos

### Hallazgos Principales:
- Xception superó a otros backbones (EfficientNetB0, ResNet50) en este dataset
- Fine-tuning + Data Augmentation mejoran el rendimiento, pero no significativamente
- Las diferentes formas y disposiciones de elementos 

### Modelos Listos para Usar:
- ✅ `Xception_augmented_finetuned.keras` - Recomendado para producción
- ✅ Funciones de predicción en `src/utils/prediction.py`
- ✅ Pipeline de evaluación completo

### Mejoras Futuras:
- Implementar ensemble de múltiples modelos
- Ajuste automático de threshold según caso de uso
- Despliegue como API REST (FastAPI)
- Testing A/B con nuevas versiones

## 11. Referencias

- **Transfer Learning:** https://keras.io/guides/transfer_learning/
- **Xception:** Chollet, F. (2017). Xception: Deep Learning with Depthwise Separable Convolutions
- **EfficientNet:** Tan & Le (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks
- **ResNet:** He et al. (2015). Deep Residual Learning for Image Recognition
- **Data Augmentation:** https://www.tensorflow.org/tutorials/images/data_augmentation